<a href="https://colab.research.google.com/github/Plee47/TC1-Rede-Neural-para-Previsao-de-Churn-com-Pipeline-Profissional-End-to-End/blob/staging/data/data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)

In [3]:
DATA_PATH = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Shape: {df_raw.shape}')
print(f'Colunas: {df_raw.columns.tolist()}')
df_raw.head()

Shape: (7043, 21)
Colunas: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
# Nulos
missing = df_raw.isnull().sum()
print('Valores nulos por coluna:')
display(pd.DataFrame({
    'nulos': missing,
    'pct (%)': (missing / len(df_raw) * 100).round(2)
})[missing > 0] if missing.sum() > 0 else pd.DataFrame({'resultado': ['Nenhum valor nulo encontrado']}))

Valores nulos por coluna:


,resultado
0,Nenhum valor nulo encontrado


In [6]:
# Brancos
df = df_raw.copy()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

problematicos = df[df['TotalCharges'].isnull()]
print(f'Registros com TotalCharges inválido: {len(problematicos)}')
display(problematicos[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])


mediana_total = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(mediana_total)
print(f'\nMediana aplicada: R$ {mediana_total:.2f}')
print(f'NaN restantes: {df["TotalCharges"].isnull().sum()}')

Registros com TotalCharges inválido: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No



Mediana aplicada: R$ 1397.47
NaN restantes: 0


In [7]:
# Duplicatas
n_dup = df.duplicated().sum()
print(f'Linhas duplicadas: {n_dup}')

# Binarizar
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)
print(f'Target binarizado: Churn=Yes → 1, Churn=No → 0')
print(df['Churn_bin'].value_counts())

Linhas duplicadas: 0
Target binarizado: Churn=Yes → 1, Churn=No → 0
Churn_bin
0    5174
1    1869
Name: count, dtype: int64


In [8]:
# Descritivas
NUM_COLS = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[NUM_COLS].describe().round(2)

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,32.37,64.76,2281.92
std,24.56,30.09,2265.27
min,0.00,18.25,18.80
25%,9.00,35.50,402.22
50%,29.00,70.35,1397.48
75%,55.00,89.85,3786.60
max,72.00,118.75,8684.80
